# ipyaidojo

> Start ipyai with a worked tooling session already in its history

In [ ]:
#| default_exp ipyaidojo

#| export
An ipyai session is an aidialog `Dialog`, saved whole as an `.ipynb` under `./.ipyai/sessions/`, and ipyai rebuilds the model's context from that dialog on every turn (`dlg2hist`). So the template needs no host-native record format: the canonical dojo dialog already has the shape ipyai resumes. What differs from the clikernel hosts is the tool. ipyai's model runs code through `py`, which runs the cell in the user's kernel and renders its outputs with aidialog's `render_outputs_ai`, so the baked round must show `py` calls with `py` results. `ipyaidojo` gets them by replaying the canonical round's cells through ipyai's own kernel client, with no model spend, and with the first bootstrap read swapped: ipyai has no `clik` skill to document. Launch is then a session write plus `ipyai -r`.

In [ ]:
#| export
import json, os, sys, tempfile
from importlib.resources import files
from fastcore.utils import *
from fastcore.script import call_parse
from aidialog.ipynb import read_ipynb, write_ipynb, reads_ipynb
from aidialog.hist import reply2dlg, dlg2reply, _parse_call
from aidialog.dialog import code_output, prompt_output
from aidialog.msg_parts import ToolResponse, tool_text
from llmdojo.tmpl import save_store, load_store, load_reg, round_gates, find_cid, doced_names, launch_config, DOJO_CANON

In [ ]:
from fastcore.test import *
import shutil
from ipyai.session import resolve_session
from llmdojo.tmpl import TMPL_PROMPT

## The ipyai round

The clikernel hosts open the round by reading their own skill, `doc(clik, pysk, edsk)`. ipyai's model never sees clikernel, so its round opens with the shared pair, and every other cell plays unchanged:

In [ ]:
#| export
TOOL = 'py'   # ipyai's model-facing tool: runs a cell in the user's kernel
BOOT_CLIK, BOOT = 'doc(clik, pysk, edsk)', 'doc(pysk, edsk)'

def ipyai_cells(
    cells, # The canonical round's kernel cell sources
):
    "The same round as ipyai plays it: the clikernel bootstrap read becomes the shared pair, since ipyai has no `clik`"
    return [BOOT if c.strip()==BOOT_CLIK else c for c in cells]

In [ ]:
test_eq(ipyai_cells(['doc(clik, pysk, edsk)', 'doc(dsk, exh, rgsk)', 'dojo_start()']), ['doc(pysk, edsk)', 'doc(dsk, exh, rgsk)', 'dojo_start()'])
ipyai_cells(['doc(clik, pysk, edsk)'])

## Replaying through ipyai

`replay_cells` plays cells the way a live ipyai session's model does: a fresh kernel owned by ipyai's `KernelSession`, seeded by `setup_tools` (which also runs the user's `~/.config/ipyai/startup.py`, so the round's imports are the real ones, just as clikernel's replay runs the clikernel startup), and each cell through the `py` tool. The result is the text the model would read back, with media tags where a cell displayed an image.

In [ ]:
#| export
async def replay_cells(
    cells, # Kernel cell sources, run in order
    cwd, # Directory to start the kernel in
    env=None, # Extra environment entries for the kernel process
):
    "Each cell's `py` result from a fresh ipyai kernel at `cwd`: the text the model reads in a live session"
    from ipyai.kernel import KernelSession
    from ipyai.bridge import setup_tools
    k = await KernelSession().start(cwd=cwd, env=env)
    try:
        _,tools = await setup_tools(k.kc)
        outs = []
        for c in cells:
            r = await tools.call_text(TOOL, dict(code=c))
            outs.append(tool_text(r.content if isinstance(r, ToolResponse) else r))
        return outs
    finally: await k.close()

A bare expression comes back as its value, a printed line as its text with the newline it printed, and a silent cell as nothing at all, exactly as `py` renders them live (this needs a running rustygate):

In [ ]:
rd = Path(tempfile.mkdtemp())
outs = await replay_cells(['1+1', "print('hi')", 'x = 3'], rd)
test_eq(outs, ['2', 'hi\n', ''])
outs

## The template store

`ipyai_dialog` turns the canonical dialog into ipyai's: every baked call is parsed back to its cell, the cells are localized to the real run dir and replayed, each call is rewritten as a `py` call with its fresh result, the round is gated, and the run dir is canonicalized again so the stored artifact reads the same everywhere. The store holds that dialog as one notebook dict, the completion id the replay earned, and the names the round documents.

In [ ]:
#| export
TMPL_DIR = files('llmdojo')/'dojo_data'/'ipyai_store'   # package data: compiled by dojobuild, shipped with the code

async def ipyai_dialog(
    src, # Path to the canonical template dialog .ipynb
    cwd, # Directory to replay in
):
    "The canonical round as ipyai plays it: calls renamed to `py`, results regenerated by replay, gated, canonicalized; returns `(dialog, outs)`"
    from llmdojo.dojo import _run_dir
    dlg = read_ipynb(str(src))
    pmsg = dlg.messages[0]
    sub = reply2dlg(pmsg)
    codes = [m for m in sub.messages if m.msg_type=='code']
    cells = ipyai_cells([_parse_call(m.content)[1]['code'].replace(DOJO_CANON, str(_run_dir())) for m in codes])   # localize: the round plays in the real run dir
    outs = await replay_cells(cells, cwd)
    for m,c,o in zip(codes, cells, outs): m.content,m.output = f'{TOOL}(code={c!r})',code_output(o)
    if probs := round_gates(cells, outs, ' '.join(m.content for m in sub.messages if m.msg_type=='note')): raise ValueError('; '.join(probs))
    pmsg.output = prompt_output(dlg2reply(sub).replace(str(_run_dir()), DOJO_CANON))   # canonicalize: the stored artifact reads the same everywhere
    dlg.meta['llmdojo'] = dict(doced=doced_names(cells))
    return dlg,outs

async def build_template(
    src, # Path to the canonical template dialog .ipynb
    d=None, # Store dir; `TMPL_DIR` if None
):
    "Replay the canonical dialog through ipyai and write the ipyai store: the dialog as a notebook dict, its completion id, and its doced list"
    with tempfile.TemporaryDirectory(prefix='ipyaidojo_') as td: dlg,outs = await ipyai_dialog(src, td)
    save_store(Path(d or TMPL_DIR), [json.loads(write_ipynb(dlg))], find_cid(outs), doced=dlg.meta['llmdojo']['doced'])
    return dlg

def load_template(
    d=None, # Store dir; `TMPL_DIR` if None
):
    "The stored template items and metadata"
    return load_store(Path(d or TMPL_DIR))

def template_dialog(
    items, # Stored template items: one notebook dict
):
    "The template dialog read back from its stored notebook dict"
    return reads_ipynb(json.dumps(items[0]))

Building from the packaged dialog replays the whole round (about a minute, no model spend), here with the llmdojo state redirected to the store dir so the replay's receipts stay out of the real registry and a parallel notebook run cannot share its run dir. The result opens with the swapped bootstrap read and calls `py` throughout, and the store reads back to the same reply:

In [ ]:
tstore = Path(tempfile.mkdtemp())
os.environ['LLMDOJO_STATE_DIR'] = str(tstore)
try: tdlg = await build_template(files('llmdojo')/'dojo_data'/'dojo_template.ipynb', tstore)
finally: del os.environ['LLMDOJO_STATE_DIR']
calls = [_parse_call(m.content) for m in reply2dlg(tdlg.messages[0]).messages if m.msg_type=='code']
test_eq({c[0] for c in calls}, {TOOL})
test_eq(calls[0][1]['code'], BOOT)
items,meta = load_template(tstore)
test_eq(template_dialog(items).messages[0].ai_res, tdlg.messages[0].ai_res)
meta

## Launching

`prep_dojo` writes the template as a session of the target project through ipyai's own `Session` (so the `.ipyai/sessions/` layout and its self-ignoring `.gitignore` are ipyai's), registers the completion id, and returns the session id that `ipyai -r` resumes. Resuming paints the round and hands the dialog to the model as real `py` tool calls.

In [ ]:
#| export
def prep_dojo(
    cwd=None, # Project to start in; the current directory if None
    d=None, # Template store dir; `TMPL_DIR` if None
):
    "Write the template as a session of `cwd` through ipyai's `Session`, register its completion id, and return the session id to resume"
    from ipyai.session import Session
    items,meta = load_reg(d, TMPL_DIR, 'ipyaidojo')
    s = Session(root=cwd or '.')
    s.save(template_dialog(items))
    return s.path.stem

Against temporary dirs, with the llmdojo state redirected so the registration stays off the real completion record:

In [ ]:
mproj = Path(tempfile.mkdtemp())
os.environ['LLMDOJO_STATE_DIR'] = str(tstore)
try: psid = prep_dojo(mproj, tstore)
finally: del os.environ['LLMDOJO_STATE_DIR']
back = read_ipynb(resolve_session(psid, mproj))
test_eq(back.messages[0].content, TMPL_PROMPT)
test_eq(back.messages[0].ai_res, tdlg.messages[0].ai_res)
psid

The launcher is a `call_parse` CLI: `--sid` prints the prepared id instead of launching, unrecognized flags are forwarded to `ipyai`, and standing arguments come from the `ipyai_args` list in `$XDG_CONFIG_HOME/ipyaidojo/config.toml` through `launch_config('ipyai')`.

In [ ]:
#| export
@call_parse(nested=True)
def main(
    sid:bool=False, # Print the prepared session id instead of launching ipyai
):
    "Prepare an ipyai session opening with the worked round and launch `ipyai -r` on it; template maintenance is `dojobuild`"
    s = prep_dojo()
    if sid: return print(s)
    os.execvp('ipyai', ['ipyai', *launch_config('ipyai'), '-r', s, *sys.argv[1:]])

## Cleanup

In [ ]:
for p in (rd, tstore, mproj): shutil.rmtree(p)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()